# Chapter 14: RAG Evaluation End-to-End
## Topic 3: Running RAGAS on Your Pipeline

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Topic 1 built each RAGAS metric's formula in isolation, tested on single, hand-picked examples. Topic 2 built a real (if small) eval set from this project's actual FD corpus. This topic combines them: running all four metrics across every entry in that eval set, against this project's actual retrieval pipeline (Chapter 7's hybrid BM25+dense approach), producing the first genuine, aggregated RAGAS evaluation of this specific project's actual RAG behavior.
- The core operational shift this topic represents: moving from "does this metric's formula work correctly on one example" (Topic 1's concern) to "what does this project's actual pipeline score, in aggregate, across a real, representative set of its own actual queries" (this topic's concern) — the same operational shift Chapter 7 Topic 9 made when it moved from explaining Recall@K's formula to actually running it across a real evaluation set.
- This topic also surfaces something genuinely new relative to Topic 1's isolated tests: running the *same* four metrics across *multiple* entries produces a distribution, not a single number — and a distribution immediately raises questions Topic 1's single-example tests couldn't: is the pipeline's faithfulness score consistently high, or high on average but with a few concerning low outliers? This is precisely the same aggregate-vs-segmented tension Chapter 9 Topic 7 raised for hallucination-rate measurement, now applied to RAGAS's four metrics specifically.


### 2. Internal Working — Step by Step

**Running RAGAS across an eval set, step by step:**

1. **For each entry in Topic 2's eval set, run this project's actual retrieval pipeline** (a real BM25 index over this project's actual FD knowledge base content, standing in for Chapter 7's full hybrid pipeline) using the entry's extracted question — this produces the retrieved chunks that context precision and context recall will be computed against.
2. **Construct a generated answer from those retrieved chunks** — since this notebook has no live LLM to call, this step uses a simple, honest template-based generation standing in for what Chapter 8's actual generation layer would produce, clearly labeled as a simplification rather than a real generation call.
3. **Compute all four RAGAS metrics (Topic 1's implementations) for each entry**: faithfulness and answer relevancy against the generated answer, context precision against the retrieved chunks and question, and context recall against the retrieved chunks and the entry's verified reference answer.
4. **Aggregate each metric separately across the entire eval set** — following Chapter 9 Topic 7's explicit warning against collapsing multiple, distinct signals into one aggregate number, this topic reports all four metrics' distributions independently, never blended into a single "RAGAS score."
5. **Inspect any entries with surprisingly low scores individually**, exactly the same "don't just look at the aggregate, look at the specific failing cases" discipline Chapter 9 Topic 7 and Chapter 10 Topic 7 both already established — a low score on one specific entry is a concrete, actionable debugging lead, not just a number to average away.


### 3. How This Is Implemented in This Project

- This notebook builds a real, small BM25 index (mirroring Chapter 7 Topic 1's approach) over this project's actual FD policy content (the same penalty, senior-citizen, and statement facts used throughout this notebook series), and runs Topic 2's eval-set questions against it for real, rather than using pre-scripted, hand-picked retrieval results.
- Given no live LLM access, the "generated answer" construction step is a simple, transparent template (concatenating the top retrieved chunk's content with the question) rather than genuine language-model generation — this is an honest limitation of what can be demonstrated in this specific environment, not a claim that this represents Chapter 8's actual, considerably more sophisticated generation layer.
- The resulting per-entry and aggregate scores from this exercise are illustrative of the *methodology* — how to run and interpret RAGAS metrics across a real eval set — rather than being treated as this project's actual, validated production RAGAS baseline, which would require the real eval set (Topic 2's necessarily small demonstration set expanded considerably) and a real generation call.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Running RAGAS's real, LLM-judge-based metrics across even a modestly-sized eval set incurs real, multiplying API cost** — each of the four metrics may require multiple LLM calls per entry (claim decomposition, per-claim entailment judgment, question generation for answer relevancy), meaning a production-scale evaluation run (hundreds or thousands of entries, per Chapter 7 Topic 9's scale concerns) needs real cost budgeting, not an assumption that evaluation itself is free.
- **A low aggregate score on any one metric requires exactly the same layered attribution discipline this notebook series has repeatedly required**: a low aggregate context precision score points at retrieval (Chapter 7's pipeline), while a low aggregate faithfulness score points at generation (Chapter 8's pipeline) — running all four together, rather than just one, is precisely what makes this attribution possible without additional, separate diagnostic work.
- **This project's actual small eval set (Topic 2) is far too small to produce a stable, trustworthy aggregate score** — exactly the same small-sample-size caution already raised repeatedly in this notebook series; the numbers this topic's exercise produces should be read as a methodology demonstration, not a genuine, final verdict on this project's RAG quality.
- **Debugging a specific low-scoring entry requires inspecting that entry's actual retrieved chunks, generated answer, and reference answer directly** — rather than treating a low RAGAS score as self-explanatory, the same "go look at the actual data" discipline Chapter 9 Topic 7's own scenario-based investigation already modeled.
- **Monitoring:** if this exercise were extended into genuine, ongoing production evaluation, tracking each of the four metrics' distributions over time (not just a point-in-time snapshot) would reveal whether pipeline changes (new chunking strategy, new prompt, new retrieval configuration) are improving or regressing specific layers — directly enabling Topic 5's A/B testing methodology.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Running all four metrics together vs running them separately, on separate schedules:** running them together (this topic's approach) gives a complete, simultaneous picture of both retrieval and generation quality for the same set of queries, at the cost of higher total LLM-call cost per evaluation pass. Running context precision/recall (retrieval-focused) more frequently than faithfulness/relevancy (generation-focused), or vice versa, could reduce cost if one layer changes more often than the other in practice — a genuine scheduling optimization worth considering once evaluation cost becomes a measured concern.
- **Reporting per-entry scores vs only the aggregate:** per-entry reporting (this topic's approach) enables the specific-failing-case investigation discipline already established elsewhere in this notebook series, at the cost of a more verbose evaluation report — the right default for any evaluation run intended to actually inform debugging, rather than just produce a single headline number.
- **How to handle an entry where retrieval itself completely fails** (returns nothing genuinely relevant): context precision and recall will correctly score very low for such an entry, but faithfulness and answer relevancy depend on whatever the generation step does with that poor context — a generation step that correctly declines to answer (Chapter 9 Topic 6's insufficient-context instruction) should arguably not be penalized by a low faithfulness score the same way a generation step that hallucinates from poor context should be — this nuance may require inspecting these specific cases manually rather than trusting the aggregate metric alone.


### 6. Alternatives and When to Use Each

- **Running all four RAGAS metrics together across a full eval set (this topic's approach):** the right choice for a comprehensive, periodic evaluation pass intended to inform broad pipeline health and guide debugging.
- **Running only context precision/recall, or only faithfulness/relevancy, on a more frequent cadence:** worth considering as a cost-saving measure if one layer (retrieval vs generation) is known to change more frequently than the other in a specific project's actual development pattern.
- **Running a cheaper, offline proxy version of these metrics (as this notebook's own necessarily offline exercise demonstrates) for frequent regression checks, reserving the full, real LLM-judge-based version for periodic, careful evaluation:** mirrors exactly Chapter 8 Topic 4's own tiered-verification philosophy, applied here to evaluation cost management specifically.


### 7. Common Mistakes and Production Failures

- Running RAGAS's real, LLM-judge-based metrics across a large eval set without budgeting for the real, multiplying API cost this incurs, especially given several metrics require multiple LLM calls per single evaluated entry.
- Reporting only a single, blended aggregate score rather than all four metrics separately, hiding exactly which layer (retrieval vs generation) is responsible for any measured quality problem.
- Treating a small, demonstration-scale eval set's aggregate scores as a genuine, trustworthy verdict on production RAG quality, rather than recognizing the same small-sample-size instability already raised repeatedly in this notebook series.
- Not inspecting specific low-scoring entries individually, missing concrete, actionable debugging information that only manual inspection of the actual retrieved chunks and generated answer can reveal.
- Penalizing a generation step's low faithfulness score without checking whether it was actually a case of poor retrieval that generation correctly and appropriately declined to hallucinate around, rather than a genuine generation-layer failure.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What's the difference between Topic 1's metric implementations and this topic's exercise?
  A: Topic 1 tested each RAGAS metric's formula in isolation, on single, hand-picked examples, to validate the formula itself works correctly. This topic runs all four metrics together across every entry in a real eval set (Topic 2), against this project's actual retrieval pipeline, producing an aggregated evaluation of the pipeline's real behavior rather than a formula-correctness check.

- Q: Why does running RAGAS across multiple entries surface a genuinely new consideration Topic 1's single-example tests didn't raise?
  A: Multiple entries produce a distribution of scores, not a single number — this immediately raises the question of whether a metric is consistently high or high-on-average-with-concerning-outliers, exactly the aggregate-vs-segmented tension Chapter 9 Topic 7 already raised for hallucination-rate measurement, now applying to RAGAS's four metrics specifically.

**Intermediate**

- Q: Why should all four RAGAS metrics be reported separately, rather than combined into one blended score, when evaluating this project's pipeline?
  A: Each metric attributes to a different layer — context precision and recall to retrieval, faithfulness and answer relevancy to generation. A blended score would hide which specific layer is responsible for any measured quality problem, exactly the same collapsed-aggregate mistake Chapter 9 Topic 7 already warned against for hallucination rate specifically.

- Q: Why is inspecting individual low-scoring entries important, beyond looking at the aggregate scores?
  A: A low aggregate score alone doesn't reveal what actually went wrong for a specific case — inspecting the actual retrieved chunks, generated answer, and reference answer for a low-scoring entry provides concrete, actionable debugging information that the aggregate number alone cannot, exactly the same "go look at the actual data" discipline already modeled in Chapter 9 Topic 7's scenario-based investigations.

**Advanced**

- Q: Design a cost-conscious evaluation schedule for running RAGAS's real, LLM-judge-based metrics on an ongoing basis, given their real, multiplying per-entry API cost.
  A: Run a cheaper, offline proxy version (like this notebook's own approach) frequently, for fast, low-cost regression detection after every pipeline change. Reserve the full, real LLM-judge-based version for periodic, more careful evaluation passes — before a major release, or on a representative sample rather than the full eval set every time — mirroring exactly Chapter 8 Topic 4's tiered-verification philosophy, now applied to evaluation cost rather than production verification cost.

- Q: A specific eval-set entry scores very low on faithfulness. Before concluding this is a generation-layer bug, what would you check first?
  A: Check whether this entry's retrieval step actually found relevant context at all — if retrieval genuinely failed to find anything useful, a low faithfulness score might actually reflect the generation step correctly declining to answer confidently from poor context (Chapter 9 Topic 6's insufficient-context instruction), rather than the generation step genuinely hallucinating from context it did have. This distinction matters because the fix differs entirely: improving retrieval versus fixing a generation-layer grounding problem.

**Scenario-based**

- Q: After running RAGAS across your full eval set, context precision scores are uniformly high, but faithfulness scores show high average with several concerning low outliers. Walk through your investigation.
  A: Given retrieval (context precision) is uniformly strong, the outlier faithfulness failures are likely generation-layer issues specific to particular query types, not a retrieval problem — inspect the specific low-faithfulness entries directly, checking whether their generated answers contain claims not supported by the (evidently good, per the high context precision score) retrieved context. This pattern — good retrieval, inconsistent generation — points investigation toward generation-layer prompting or model behavior (Chapter 9 Topic 6's RAG-specific prompting) rather than retrieval tuning.

**Follow-up questions to expect:**

- "How would you decide when your eval set is large enough to trust the aggregate scores it produces?"
- "What would you change about this evaluation methodology once a real generation layer (not this notebook's template stand-in) is available to test?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **Running the same evaluation methodology repeatedly, at different points in a pipeline's evolution, is what transforms a one-time metric computation into a genuine evaluation practice** — this topic's single run is a demonstration of mechanics; genuine evaluation practice (Chapter 10 Topic 7's discipline) requires this same process repeated consistently over time, with results saved and compared, not run once and forgotten.
- **The distinction between "testing that a metric's formula works" (Topic 1) and "using that metric to evaluate a real system" (this topic) is a general pattern in any evaluation methodology** — unit-testing a metric's implementation is a genuinely different activity from applying that validated metric to draw real conclusions about a system's actual behavior, and conflating the two risks either under-testing the metric itself or over-trusting a single evaluation run's results.
- **This topic directly enables Topic 5's A/B testing**: the exact process built here — run the pipeline against the eval set, compute all four metrics, aggregate and inspect — is precisely what Topic 5 repeats for two different retrieval configurations, holding this topic's methodology constant while varying only the configuration under comparison.

### 10. Quick Revision Summary (for last-minute interview prep)

> Running RAGAS on a real pipeline means combining Topic 1's validated metric implementations with Topic 2's real eval set, running this project's actual retrieval pipeline against every eval-set entry, computing all four metrics per entry, and aggregating each separately — never blended into one score, mirroring Chapter 9 Topic 7's explicit warning against collapsed aggregates. Multiple entries produce a distribution, not a single number, raising the same aggregate-vs-outlier tension already established for hallucination-rate measurement — a consistently high score is meaningfully different from a high average hiding concerning individual failures, and specific low-scoring entries deserve direct inspection, not just aggregate reporting. Every RAGAS metric requires real LLM calls at real, multiplying cost across an eval set, motivating the same tiered-verification cost strategy already established elsewhere in this notebook series: cheap, offline proxies for frequent checks, full LLM-judge versions reserved for periodic, careful evaluation. This exercise demonstrates the methodology concretely using this project's real FD content and Topic 2's real (if small and explicitly limited) eval set — the numbers produced illustrate the process, not a final, trustworthy verdict on this project's actual production RAG quality, which would require a considerably larger eval set and genuine LLM-based generation.


### Module 1: The Real Pipeline — BM25 Retrieval Plus Template Generation

Build a real BM25 index over this project's actual FD policy content, and a simple, honest template-based generation step standing in for a real LLM call.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Real retrieval pipeline + honest template generation
# ------------------------------------------------------------------

from rank_bm25 import BM25Okapi
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

def tokenize(text: str) -> list:
    return text.lower().split()

def normalize_vector(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def cosine_similarity(a, b):
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / d) if d > 0 else 0.0

def embed_texts(texts: list, n_components: int = 3) -> np.ndarray:
    vectorizer = TfidfVectorizer()
    sparse = vectorizer.fit_transform(texts)
    n_comp = min(n_components, sparse.shape[1] - 1, len(texts) - 1)
    if n_comp < 1:
        n_comp = 1
    svd = TruncatedSVD(n_components=n_comp, random_state=42)
    dense = svd.fit_transform(sparse)
    return np.array([normalize_vector(v) for v in dense])

def extract_claims(text: str) -> list:
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]

def simulate_entailment_judgment(claim: str, context: str) -> bool:
    claim_words = set(w for w in claim.lower().split() if len(w) > 3)
    context_words = set(w for w in context.lower().split() if len(w) > 3)
    if not claim_words:
        return True
    overlap_ratio = len(claim_words & context_words) / len(claim_words)
    return overlap_ratio >= 0.5


# this project's REAL FD policy content, reused throughout this notebook series
KNOWLEDGE_BASE = [
    "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
    "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
    "FD account statements including current balance and accrued interest are available on request at any branch or via the customer portal.",
    "Nomination facility is available for all Fixed Deposit account holders at no charge.",
]
tokenized_corpus = [tokenize(doc) for doc in KNOWLEDGE_BASE]
bm25 = BM25Okapi(tokenized_corpus)

# Topic 2's real eval set entries (reconstructed here for this notebook's
# self-contained execution)
EVAL_SET = [
    {"question": "What is the penalty for premature FD withdrawal?",
     "reference_answer": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate."},
    {"question": "Do senior citizens get extra interest on FDs?",
     "reference_answer": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures."},
    {"question": "How can I get my FD account statement?",
     "reference_answer": "FD account statements including current balance and accrued interest are available on request at any branch or via the customer portal."},
]

def retrieve(query: str, top_k: int = 2) -> list:
    """REAL retrieval -- BM25, this project's actual chosen approach
    (Chapter 7 Topic 1), not a scripted or pre-picked result."""
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(range(len(KNOWLEDGE_BASE)), key=lambda i: scores[i], reverse=True)
    return [KNOWLEDGE_BASE[i] for i in ranked[:top_k]]


def generate_answer(question: str, retrieved_chunks: list) -> str:
    """HONEST template-based generation -- standing in for a real LLM
    call, which is not available in this environment. Simply uses the
    top retrieved chunk directly, clearly a simplification."""
    if not retrieved_chunks:
        return "I don't have information to answer this question."
    return retrieved_chunks[0]


print("=" * 70)
print("MODULE 1: REAL PIPELINE READY")
print("=" * 70)
print(f"Knowledge base: {len(KNOWLEDGE_BASE)} real FD policy chunks")
print(f"Eval set: {len(EVAL_SET)} entries (from Topic 2's real, verified extraction)")

for entry in EVAL_SET:
    retrieved = retrieve(entry["question"])
    answer = generate_answer(entry["question"], retrieved)
    entry_question = entry["question"]
    print(f"\nQuestion: '{entry_question}'")
    print(f"  Retrieved top chunk: '{retrieved[0][:60]}...'")
    print(f"  Generated answer: '{answer[:60]}...'")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: REAL PIPELINE READY
Knowledge base: 4 real FD policy chunks
Eval set: 3 entries (from Topic 2's real, verified extraction)

Question: 'What is the penalty for premature FD withdrawal?'
  Retrieved top chunk: 'Nomination facility is available for all Fixed Deposit accou...'
  Generated answer: 'Nomination facility is available for all Fixed Deposit accou...'

Question: 'Do senior citizens get extra interest on FDs?'
  Retrieved top chunk: 'Senior citizens receive an additional 0.5 percent interest o...'
  Generated answer: 'Senior citizens receive an additional 0.5 percent interest o...'

Question: 'How can I get my FD account statement?'
  Retrieved top chunk: 'Premature withdrawal of FD incurs a 1 percent penalty on the...'
  Generated answer: 'Premature withdrawal of FD incurs a 1 percent penalty on the...'

Module 1 complete. Run Module 2 next.


### Module 2: Running All Four RAGAS Metrics Across the Full Eval Set

Compute faithfulness, answer relevancy, context precision, and context recall for every eval-set entry, reporting each metric separately -- never collapsed into one aggregate.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: All four RAGAS metrics, run across the FULL eval set
# ------------------------------------------------------------------

def compute_faithfulness(answer: str, context: str) -> float:
    claims = extract_claims(answer)
    if not claims:
        return 0.0
    entailed = [simulate_entailment_judgment(c, context) for c in claims]
    return sum(entailed) / len(claims)

def compute_answer_relevancy(answer: str, actual_question: str) -> float:
    simulated_question = "what is " + " ".join(answer.lower().split()[:8])
    vectors = embed_texts([actual_question, simulated_question])
    return cosine_similarity(vectors[0], vectors[1])

def compute_context_precision(question: str, retrieved_chunks: list) -> float:
    relevance_flags = [simulate_entailment_judgment(question, c) for c in retrieved_chunks]
    precisions_at_k = []
    relevant_so_far = 0
    for k, is_relevant in enumerate(relevance_flags, start=1):
        if is_relevant:
            relevant_so_far += 1
            precisions_at_k.append(relevant_so_far / k)
    return sum(precisions_at_k) / len(precisions_at_k) if precisions_at_k else 0.0

def compute_context_recall(reference_answer: str, retrieved_chunks: list) -> float:
    combined_context = " ".join(retrieved_chunks)
    claims = extract_claims(reference_answer)
    if not claims:
        return 0.0
    attributable = [simulate_entailment_judgment(c, combined_context) for c in claims]
    return sum(attributable) / len(claims)


print("=" * 70)
print("MODULE 2: ALL FOUR RAGAS METRICS, PER ENTRY (never collapsed)")
print("=" * 70)

results = []
for entry in EVAL_SET:
    question = entry["question"]
    reference = entry["reference_answer"]
    retrieved_chunks = retrieve(question)
    generated_answer = generate_answer(question, retrieved_chunks)
    combined_context = " ".join(retrieved_chunks)

    faithfulness = compute_faithfulness(generated_answer, combined_context)
    relevancy = compute_answer_relevancy(generated_answer, question)
    precision = compute_context_precision(question, retrieved_chunks)
    recall = compute_context_recall(reference, retrieved_chunks)

    results.append({"question": question, "faithfulness": faithfulness,
                     "answer_relevancy": relevancy, "context_precision": precision,
                     "context_recall": recall})

    print(f"\nQuestion: '{question}'")
    print(f"  Faithfulness:      {faithfulness:.3f}")
    print(f"  Answer relevancy:  {relevancy:.3f}")
    print(f"  Context precision: {precision:.3f}")
    print(f"  Context recall:    {recall:.3f}")

print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: ALL FOUR RAGAS METRICS, PER ENTRY (never collapsed)

Question: 'What is the penalty for premature FD withdrawal?'
  Faithfulness:      1.000
  Answer relevancy:  1.000
  Context precision: 0.500
  Context recall:    1.000

Question: 'Do senior citizens get extra interest on FDs?'
  Faithfulness:      1.000
  Answer relevancy:  1.000
  Context precision: 1.000
  Context recall:    1.000

Question: 'How can I get my FD account statement?'
  Faithfulness:      1.000
  Answer relevancy:  1.000
  Context precision: 0.000
  Context recall:    0.000

Module 2 complete. Run Module 3 next.


### Module 3: Aggregating and Inspecting — Never Just One Number

Aggregate each metric separately across the eval set, and specifically inspect the lowest-scoring entry for each metric -- concrete, actionable debugging information the aggregate alone can't provide.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Aggregating (separately) + inspecting specific low scores
# ------------------------------------------------------------------

metric_names = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

print("=" * 70)
print("MODULE 3: AGGREGATE SCORES -- REPORTED SEPARATELY, NEVER BLENDED")
print("=" * 70)

aggregates = {}
for metric in metric_names:
    values = [r[metric] for r in results]
    avg = sum(values) / len(values)
    aggregates[metric] = avg
    print(f"\n{metric}:")
    print(f"  Per-entry scores: {[round(v, 3) for v in values]}")
    print(f"  Average: {avg:.3f}")

print("\n" + "=" * 70)
print("INSPECTING THE LOWEST-SCORING ENTRY PER METRIC")
print("=" * 70)

for metric in metric_names:
    lowest_entry = min(results, key=lambda r: r[metric])
    print(f"\nLowest {metric} score: {lowest_entry[metric]:.3f}")
    lowest_question = lowest_entry["question"]
    print(f"  Question: '{lowest_question}'")
    if lowest_entry[metric] < 1.0:
        print(f"  -> Worth manually inspecting this specific case's retrieved")
        print(f"     chunks and generated answer directly, not just noting the score.")
    else:
        print(f"  -> Even the LOWEST score for this metric is perfect (1.0) --")
        print(f"     this small eval set may be too easy/small to surface real")
        print(f"     weaknesses, exactly the small-sample-size caution this")
        print(f"     topic's theory raises.")

print("\n" + "=" * 70)
print("HONEST CONCLUSION")
print("=" * 70)
print(f"""
This exercise demonstrates the METHODOLOGY of running RAGAS across a
real eval set against a real (BM25) retrieval pipeline -- it does NOT
represent a genuine, trustworthy production RAGAS baseline for this
project, given:
  - Only {len(EVAL_SET)} eval-set entries (Chapter 7 Topic 9's small-sample warning)
  - Template-based generation, not a real LLM call
  - Offline entailment/similarity proxies, not real LLM-judge calls

A genuine production baseline needs Topic 2's eval set expanded
considerably, run against this project's REAL hybrid retrieval
pipeline (Chapter 7) and REAL generation layer (Chapter 8), using
REAL LLM-judge calls for each metric.
""")

print("Module 3 complete. All Chapter 14 Topic 3 modules done.")
print()
print("=" * 70)
print("CHAPTER 14 TOPIC 3 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  All 4 RAGAS metrics computed PER ENTRY across the full eval set,
  then aggregated SEPARATELY -- never blended into one score.

  Multiple entries produce a DISTRIBUTION -- inspecting the lowest-
  scoring entry PER METRIC gives concrete, actionable debugging
  information the aggregate alone cannot provide.

  This exercise is a METHODOLOGY demonstration using real project
  content and a real (BM25) pipeline -- explicitly NOT a trustworthy
  production baseline, given the small eval set, template generation,
  and offline metric proxies used here.

  This exact process (run pipeline -> compute 4 metrics -> aggregate
  separately -> inspect outliers) is precisely what Topic 5's A/B
  test repeats for two different retrieval configurations.
""")


MODULE 3: AGGREGATE SCORES -- REPORTED SEPARATELY, NEVER BLENDED

faithfulness:
  Per-entry scores: [1.0, 1.0, 1.0]
  Average: 1.000

answer_relevancy:
  Per-entry scores: [1.0, 1.0, 1.0]
  Average: 1.000

context_precision:
  Per-entry scores: [0.5, 1.0, 0.0]
  Average: 0.500

context_recall:
  Per-entry scores: [1.0, 1.0, 0.0]
  Average: 0.667

INSPECTING THE LOWEST-SCORING ENTRY PER METRIC

Lowest faithfulness score: 1.000
  Question: 'What is the penalty for premature FD withdrawal?'
  -> Even the LOWEST score for this metric is perfect (1.0) --
     this small eval set may be too easy/small to surface real
     weaknesses, exactly the small-sample-size caution this
     topic's theory raises.

Lowest answer_relevancy score: 1.000
  Question: 'What is the penalty for premature FD withdrawal?'
  -> Even the LOWEST score for this metric is perfect (1.0) --
     this small eval set may be too easy/small to surface real
     weaknesses, exactly the small-sample-size caution this
     t